In [1]:
from repeaters_NV import RepeaterParams, QuantumRepeaterNetwork, build_s2_graph, build_graph
import numpy as np
import networkx as nx
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
from copy import deepcopy

# Optional: Set seaborn style for publication-ready plots
sns.set_theme(style="whitegrid")


In [14]:
params = RepeaterParams()
params.p_det   = 0.95
params.alpha   = 0.18
params.V       = 0.95
params.nu      = 10e6
params.R_dark  = 100
params.delta_det = 100e-12
params.eta_c   = 0.8
params.P_BSM   = 0.9993
params.eta_M   = 0.46
params.T_coh   = 10

A, dist, coords = build_s2_graph(N=1000, beta=2.6261, mu=0.0233, scale='country')

net = QuantumRepeaterNetwork(params, A, dist, architecture='node')
df  = net.analyze_all_paths(source=0)

In [15]:
display(df)

,R,Q,SKR,path,F
526,172273.291112,0.016667,65068.858547,"[0, 526]",0.975000
161,118297.994042,0.032778,34526.603397,"[0, 526, 161]",0.950833
155,117608.314786,0.032778,34325.310673,"[0, 526, 155]",0.950833
294,116373.687020,0.032778,33964.966973,"[0, 526, 294]",0.950833
286,114864.814551,0.032778,33524.581079,"[0, 526, 286]",0.950833
...,...,...,...,...,...
818,192.216047,0.105838,2.454040,"[0, 526, 733, 268, 299, 569, 627, 818]",0.841243
939,1121.928896,0.105666,14.919535,"[0, 526, 369, 95, 269, 915, 768, 939]",0.841502
491,2089.920650,0.118794,-54.094488,"[0, 526, 213, 459, 953, 648, 546, 597, 491]",0.821809
864,33.241724,0.093269,1.752246,"[0, 526, 733, 268, 299, 517, 864]",0.860097


In [4]:
def plot_skr_vs_pbsm(nets_dict):
    """
    nets_dict: dictionary of { 'Label': QuantumRepeaterNetwork_instance }
    """
    plt.figure(figsize=(8, 6))
    pbsm_values = np.linspace(0.1, 1.0, 20)

    for label, net in nets_dict.items():
        # 1. Get all paths
        df = net.analyze_all_paths(source=0)

        # 2. Filter for paths with at least 2 hops (1 repeater)
        df['hops'] = df['path'].apply(lambda x: len(x) - 1)
        df_multi_hop = df[df['hops'] >= 2]

        if df_multi_hop.empty:
            continue

        # 3. Get the path with the highest SKR
        best_row = df_multi_hop.loc[df_multi_hop['SKR'].idxmax()]
        best_path = best_row['path']

        skr_results = []
        original_pbsm = net.params.P_BSM

        # 4. Vary P_BSM and recalculate SKR for this specific path
        for pbsm in pbsm_values:
            net.params.P_BSM = pbsm
            skr = net.secret_key_rate(best_path)
            skr_results.append(skr if skr > 0 else 0)

        # Restore original P_BSM
        net.params.P_BSM = original_pbsm

        plt.plot(pbsm_values, skr_results, marker='o', label=f"{label} ({len(best_path)-1} hops)")

    plt.xlabel("Bell State Measurement Probability (P_BSM)")
    plt.ylabel("Secret Key Rate (bits/s)")
    plt.title("Optimal Path Performance vs P_BSM")
    plt.yscale('log')
    plt.legend()
    plt.show()

In [5]:
def plot_qber_vs_hops_no_loss(net):
    original_alpha = net.params.alpha
    net.params.alpha = 0.0  # Temporarily remove distance penalty

    # Rebuild probability matrices with alpha=0
    net.Probs_mtx, net.eta_A_mtx, net.eta_B_mtx = net._build_prob_matrix()

    df = net.analyze_all_paths(source=0)
    df['hops'] = df['path'].apply(lambda x: len(x) - 1)

    # Since alpha=0, all paths with N hops have identical QBER.
    # We just drop duplicates to get one QBER value per hop count.
    df_unique_hops = df.drop_duplicates(subset=['hops']).sort_values('hops')

    plt.figure(figsize=(8, 6))
    plt.plot(df_unique_hops['hops'], df_unique_hops['Q'] * 100, marker='s', color='purple')

    # Add the 11% death threshold
    plt.axhline(11.0, linestyle='--', color='red', label="11% Death Threshold")

    plt.xlabel("Number of Hops")
    plt.ylabel("QBER (%)")
    plt.title("Theoretical QBER Limit vs Hops (0 dB/km Fiber Loss)")
    plt.xticks(df_unique_hops['hops'])
    plt.legend()
    plt.show()

    # Restore original alpha and rebuild matrices
    net.params.alpha = original_alpha
    net.Probs_mtx, net.eta_A_mtx, net.eta_B_mtx = net._build_prob_matrix()

In [6]:
def plot_repeater_payoff_synthetic(params, total_distances=[10, 50, 100], max_hops=10):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    hop_range = np.arange(1, max_hops + 1)

    for D in total_distances:
        qber_list, skr_list = [], []

        for hops in hop_range:
            d_link = D / hops  # Equal distance per hop

            # 1. Transmission (assuming 'node' architecture: source at A)
            eta_A = params.eta_c * params.p_det
            eta_B = params.eta_c * params.p_det * 10**(-params.alpha * d_link / 10)
            P_ent = params.p_pair * eta_A * eta_B

            # 2. Sequential Time
            T = 1.0 / P_ent
            for _ in range(2, hops + 1):
                T = (T + (1.0 / P_ent)) / params.P_BSM
            R_ent = 0.5 * params.nu / T

            # 3. QBER
            p_dc = params.R_dark * params.delta_det
            p_acc = (params.p_pair * eta_A * (1 - eta_B) * p_dc +
                     params.p_pair * eta_B * (1 - eta_A) * p_dc + p_dc**2)
            p_true = params.p_pair * eta_A * eta_B

            Q_link = (p_true * (params.q_0 + params.p_pair / 2) + 0.5 * p_acc) / (p_true + p_acc)

            # Werner state accumulation
            W = 1.0
            for _ in range(hops):
                W *= (1 - 2 * Q_link)
            Q_tot = (1 - W) / 2

            # 4. SKR
            if Q_tot == 0 or Q_tot == 1:
                H = 0
            else:
                H = -Q_tot * np.log2(Q_tot) - (1 - Q_tot) * np.log2(1 - Q_tot)

            SKR = R_ent * (1 - 2 * H)

            qber_list.append(Q_tot * 100)
            skr_list.append(SKR if SKR > 0 else 0)

        ax1.plot(hop_range, qber_list, marker='o', label=f"Total Dist: {D} km")
        ax2.plot(hop_range, skr_list, marker='o', label=f"Total Dist: {D} km")

    # QBER Plot formatting
    ax1.axhline(11.0, linestyle='--', color='red', label="11% Threshold")
    ax1.set_xlabel("Number of Hops")
    ax1.set_ylabel("Total QBER (%)")
    ax1.set_title("QBER vs Hops for Fixed Distances")
    ax1.legend()

    # SKR Plot formatting
    ax2.axhline(0.0, linestyle='--', color='red')
    ax2.set_yscale('log')
    ax2.set_xlabel("Number of Hops")
    ax2.set_ylabel("Secret Key Rate (bits/s)")
    ax2.set_title("SKR vs Hops for Fixed Distances")
    ax2.legend()

    plt.tight_layout()
    plt.show()

In [7]:
def plot_skr_vs_distance_comparison(nets_dict):
    plt.figure(figsize=(9, 6))

    for label, net in nets_dict.items():
        df = net.analyze_all_paths(source=0)
        # Calculate total physical distance of each path
        df['total_dist'] = df['path'].apply(lambda x: sum(net.path_distances(x)))

        # Filter out negative SKRs
        df_valid = df[df['SKR'] > 0]

        # Scatter plot
        plt.scatter(df_valid['total_dist'], df_valid['SKR'], alpha=0.6, label=label, edgecolors='w')

    plt.yscale('log')
    plt.xlabel("Total Path Distance (km)")
    plt.ylabel("Secret Key Rate (bits/s)")
    plt.title("Performance Decay over Geographic Distance")
    plt.legend()
    plt.show()

In [8]:
def plot_spatial_heatmaps(net_s2, net_custom):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

    def draw_network_heatmap(net, ax, title, use_spring_layout=False):
        df = net.analyze_all_paths(source=0)

        # 1. Get Coordinates
        if use_spring_layout or net.coords is None:
            # Generate dummy coordinates using the distance matrix as spring weights
            G = nx.from_numpy_array(net.A)
            pos = nx.spring_layout(G, weight=net.dist)
            x = np.array([pos[i][0] for i in range(len(G.nodes))])
            y = np.array([pos[i][1] for i in range(len(G.nodes))])
        else:
            # Import your local sphere projection logic
            from sequential_repeaters import _sphere_to_2d
            x, y = _sphere_to_2d(net.coords, net.scale_km)

        # 2. Draw edges
        rows, cols = np.where(np.triu(net.A, k=1) > 0)
        for i, j in zip(rows, cols):
            ax.plot([x[i], x[j]], [y[i], y[j]], color='gray', alpha=0.2, linewidth=0.5, zorder=1)

        # 3. Map SKR to node colors
        skr_values = np.zeros(len(x))
        for dest in df.index:
            skr_values[dest] = df.loc[dest, 'SKR']

        # Log scale mapping for positive SKRs (0 gets gray/red)
        skr_pos = skr_values[skr_values > 0]
        norm = mcolors.LogNorm(vmin=skr_pos.min() if len(skr_pos)>0 else 1e-10,
                               vmax=skr_pos.max() if len(skr_pos)>0 else 1)

        colors = ['red' if v <= 0 else plt.cm.viridis(norm(v)) for v in skr_values]
        colors[0] = 'gold' # Highlight source

        ax.scatter(x, y, c=colors, s=30, zorder=2, edgecolors='k', linewidth=0.5)
        ax.set_title(title)
        ax.axis('off')

    draw_network_heatmap(net_s2, ax1, "S2 City Model Heatmap", use_spring_layout=False)
    draw_network_heatmap(net_custom, ax2, "Custom Hub-Spoke Heatmap", use_spring_layout=True)

    plt.tight_layout()
    plt.show()